# Semana 9: Automatizando el Gradiente
## Notebook Conceptual (NB1) - PyTorch y Descenso de Gradiente

### Propósito de la sesión
Liberarse de derivar a mano y comprender cómo las librerías modernas automatizan el cálculo de gradientes. Implementaremos descenso de gradiente desde cero con NumPy, luego usaremos PyTorch con `autograd` y `optim.SGD`, compararemos resultados y experimentaremos con diferentes tasas de aprendizaje y variantes del algoritmo.

### Objetivos de aprendizaje
*   Implementar **descenso de gradiente desde cero** con NumPy para regresión lineal.
*   Implementar el mismo modelo con **PyTorch** usando diferenciación automática (`autograd`).
*   Comparar la facilidad de implementación y los resultados.
*   Experimentar con diferentes **tasas de aprendizaje** ($\eta$) y observar convergencia/divergencia.
*   Introducir variantes: **Batch GD**, **Stochastic GD** (SGD), **Mini-batch GD**.
*   Conectar con aplicaciones reales: fine-tuning de LLMs (Adam) y Deep Q-Learning.

## Configuración Inicial

Importamos las librerías necesarias: NumPy para la implementación manual, PyTorch para la automática, y Matplotlib para visualizaciones.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Verificamos versión de PyTorch
print(f"PyTorch versión: {torch.__version__}")
print(f"¿CUDA disponible? {torch.cuda.is_available()}")

# Fijamos semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\n🎯 Librerías importadas correctamente.")

---
## 1. Generación de Datos Sintéticos

Generamos datos para regresión lineal con una sola característica: $y = 2 + 3x + \epsilon$

In [ ]:
# Parámetros verdaderos
true_w = 3.0
true_b = 2.0

# Generamos datos
n_samples = 100
X = np.random.randn(n_samples, 1) * 2  # característica
y = true_w * X + true_b + np.random.randn(n_samples, 1) * 1.5  # añadimos ruido

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, edgecolors='k')
plt.plot(X, true_w * X + true_b, 'r-', label=f'Recta verdadera: y = {true_w}x + {true_b}')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Datos sintéticos para regresión lineal')
plt.legend()
plt.grid(True)
plt.show()

---
## 2. Implementación Manual con NumPy

### 2.1. Función de costo y gradiente

Para regresión lineal con MSE, el gradiente tiene forma cerrada:

$$\nabla J(\mathbf{w}) = \frac{2}{n} \mathbf{X}^T (\mathbf{X}\mathbf{w} - \mathbf{y})$$

In [ ]:
# Añadimos columna de unos para el intercepto
X_b = np.c_[np.ones((n_samples, 1)), X]

def mse_gradient_numpy(X, y, w):
    """
    Calcula el gradiente del MSE.
    w: vector (intercepto, pendiente)
    """
    n = len(y)
    y_pred = X @ w
    grad = (2/n) * X.T @ (y_pred - y)
    return grad

def gradient_descent_numpy(X, y, w_init, lr=0.1, epochs=100, batch_size=None, verbose=True):
    """
    Descenso de gradiente con soporte para batch, mini-batch y SGD.
    batch_size=None: Batch GD
    batch_size=1: SGD
    batch_size=32: Mini-batch GD
    """
    w = w_init.copy()
    n = len(y)
    loss_history = []
    w_history = [w.copy()]
    
    for epoch in range(epochs):
        if batch_size is None:
            # Batch GD
            grad = mse_gradient_numpy(X, y, w)
            w = w - lr * grad
            y_pred = X @ w
            loss = np.mean((y_pred - y)**2)
            loss_history.append(loss)
            w_history.append(w.copy())
        else:
            # Mini-batch o SGD
            indices = np.random.permutation(n)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            epoch_loss = 0.0
            for i in range(0, n, batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                grad = mse_gradient_numpy(X_batch, y_batch, w)
                w = w - lr * grad
                y_pred_batch = X_batch @ w
                loss = np.mean((y_pred_batch - y_batch)**2)
                epoch_loss += loss * len(X_batch)
            loss_history.append(epoch_loss / n)
            w_history.append(w.copy())
        
        if verbose and epoch % 20 == 0:
            print(f"Epoch {epoch:3d}, Loss: {loss_history[-1]:.6f}")
    
    return w, loss_history, w_history

# Inicialización
w_init = np.array([0.0, 0.0])

# Batch GD
print("🔷 Batch Gradient Descent con NumPy:")
w_np_batch, loss_np_batch, _ = gradient_descent_numpy(
    X_b, y, w_init, lr=0.1, epochs=100, batch_size=None
)
print(f"\n✅ Pesos finales: b={w_np_batch[0]:.4f}, w={w_np_batch[1]:.4f}")

In [ ]:
# Visualización de la pérdida
plt.figure(figsize=(10, 6))
plt.plot(loss_np_batch, 'b-', linewidth=2, label='Batch GD')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Curva de aprendizaje - NumPy Batch GD')
plt.legend()
plt.grid(True)
plt.yscale('log')
plt.show()

---
## 3. Implementación con PyTorch (autograd y optim)

### 3.1. Modelo lineal con autograd

In [ ]:
# Convertimos datos a tensores de PyTorch
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

# Definimos el modelo con nn.Linear
model = nn.Linear(1, 1)

# Inicializamos con los mismos valores que en NumPy (para comparación)
with torch.no_grad():
    model.weight.fill_(0.0)
    model.bias.fill_(0.0)

# Función de pérdida y optimizador
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Entrenamiento
epochs = 100
loss_pt_batch = []

print("🔶 Batch Gradient Descent con PyTorch:")
for epoch in range(epochs):
    # Forward
    y_pred = model(X_t)
    loss = criterion(y_pred, y_t)
    loss_pt_batch.append(loss.item())
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    
    # Update
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}, Loss: {loss.item():.6f}")

with torch.no_grad():
    w_pt = model.weight.item()
    b_pt = model.bias.item()
print(f"\n✅ Pesos finales PyTorch: b={b_pt:.4f}, w={w_pt:.4f}")

### 3.2. Comparación de resultados

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(loss_np_batch, 'b-', linewidth=2, label='NumPy Batch GD')
plt.plot(loss_pt_batch, 'r--', linewidth=2, label='PyTorch Batch GD')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Comparación de curvas de pérdida')
plt.legend()
plt.grid(True)
plt.yscale('log')

plt.subplot(1, 2, 2)
plt.scatter(X, y, alpha=0.5, label='Datos')
x_line = np.linspace(X.min(), X.max(), 100)
plt.plot(x_line, w_np_batch[1] * x_line + w_np_batch[0], 'b-', linewidth=2, label=f'NumPy: y={w_np_batch[1]:.2f}x+{w_np_batch[0]:.2f}')
plt.plot(x_line, w_pt * x_line + b_pt, 'r--', linewidth=2, label=f'PyTorch: y={w_pt:.2f}x+{b_pt:.2f}')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Rectas ajustadas')
plt.legend()
plt.grid(True)

plt.suptitle('NumPy vs PyTorch: Batch Gradient Descent')
plt.tight_layout()
plt.show()

---
## 4. Experimentación con Tasas de Aprendizaje

Probamos diferentes learning rates y observamos convergencia o divergencia.

In [ ]:
learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0]
epochs = 50

plt.figure(figsize=(14, 8))

for i, lr in enumerate(learning_rates):
    # Reiniciamos el modelo
    model = nn.Linear(1, 1)
    with torch.no_grad():
        model.weight.fill_(0.0)
        model.bias.fill_(0.0)
    optimizer = optim.SGD(model.parameters(), lr=lr)
    
    loss_history = []
    for epoch in range(epochs):
        y_pred = model(X_t)
        loss = criterion(y_pred, y_t)
        loss_history.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    plt.subplot(2, 3, i+1)
    plt.plot(loss_history, linewidth=2)
    plt.xlabel('Época')
    plt.ylabel('MSE')
    plt.title(f'lr = {lr}')
    plt.grid(True)
    if lr >= 0.5:
        plt.ylim(0, 100)  # para ver divergencia

plt.suptitle('Efecto de la tasa de aprendizaje en la convergencia', fontsize=14)
plt.tight_layout()
plt.show()

print("📌 lr pequeño: convergencia lenta.")
print("📌 lr moderado: convergencia rápida y estable.")
print("📌 lr grande: divergencia (loss explota).")

---
## 5. Variantes del Descenso de Gradiente

### 5.1. Stochastic Gradient Descent (SGD) - batch_size=1

In [ ]:
# SGD con NumPy
print("🔷 Stochastic Gradient Descent con NumPy:")
w_sgd_np, loss_sgd_np, _ = gradient_descent_numpy(
    X_b, y, w_init, lr=0.1, epochs=50, batch_size=1, verbose=False
)

# SGD con PyTorch
model_sgd = nn.Linear(1, 1)
with torch.no_grad():
    model_sgd.weight.fill_(0.0)
    model_sgd.bias.fill_(0.0)
optimizer_sgd = optim.SGD(model_sgd.parameters(), lr=0.1)

dataset = TensorDataset(X_t, y_t)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

loss_sgd_pt = []
for epoch in range(50):
    epoch_loss = 0.0
    for X_batch, y_batch in dataloader:
        y_pred = model_sgd(X_batch)
        loss = criterion(y_pred, y_batch)
        optimizer_sgd.zero_grad()
        loss.backward()
        optimizer_sgd.step()
        epoch_loss += loss.item() * len(X_batch)
    loss_sgd_pt.append(epoch_loss / len(X_t))

plt.figure(figsize=(12, 5))
plt.plot(loss_sgd_np, 'b-', alpha=0.7, label='NumPy SGD')
plt.plot(loss_sgd_pt, 'r--', alpha=0.7, label='PyTorch SGD')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Stochastic Gradient Descent (batch_size=1)')
plt.legend()
plt.grid(True)
plt.show()

### 5.2. Mini-batch Gradient Descent (batch_size=16)

In [ ]:
# Mini-batch con NumPy
w_mini_np, loss_mini_np, _ = gradient_descent_numpy(
    X_b, y, w_init, lr=0.1, epochs=50, batch_size=16, verbose=False
)

# Mini-batch con PyTorch
model_mini = nn.Linear(1, 1)
with torch.no_grad():
    model_mini.weight.fill_(0.0)
    model_mini.bias.fill_(0.0)
optimizer_mini = optim.SGD(model_mini.parameters(), lr=0.1)

dataloader_mini = DataLoader(dataset, batch_size=16, shuffle=True)

loss_mini_pt = []
for epoch in range(50):
    epoch_loss = 0.0
    for X_batch, y_batch in dataloader_mini:
        y_pred = model_mini(X_batch)
        loss = criterion(y_pred, y_batch)
        optimizer_mini.zero_grad()
        loss.backward()
        optimizer_mini.step()
        epoch_loss += loss.item() * len(X_batch)
    loss_mini_pt.append(epoch_loss / len(X_t))

plt.figure(figsize=(12, 5))
plt.plot(loss_mini_np, 'b-', alpha=0.7, label='NumPy Mini-batch')
plt.plot(loss_mini_pt, 'r--', alpha=0.7, label='PyTorch Mini-batch')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Mini-batch Gradient Descent (batch_size=16)')
plt.legend()
plt.grid(True)
plt.show()

### 5.3. Comparación de variantes

In [ ]:
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(loss_np_batch, 'b-', linewidth=2, label='Batch')
plt.plot(loss_mini_np, 'g-', linewidth=2, label='Mini-batch (16)')
plt.plot(loss_sgd_np, 'r-', linewidth=2, label='SGD (1)')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Comparación de variantes (NumPy)')
plt.legend()
plt.grid(True)
plt.yscale('log')

plt.subplot(1, 2, 2)
plt.plot(loss_pt_batch, 'b-', linewidth=2, label='Batch')
plt.plot(loss_mini_pt, 'g-', linewidth=2, label='Mini-batch (16)')
plt.plot(loss_sgd_pt, 'r-', linewidth=2, label='SGD (1)')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Comparación de variantes (PyTorch)')
plt.legend()
plt.grid(True)
plt.yscale('log')

plt.suptitle('Batch vs Mini-batch vs SGD', fontsize=14)
plt.tight_layout()
plt.show()

print("📌 Batch: curva suave, lenta por época.")
print("📌 SGD: mucho ruido, pero puede escapar de mínimos locales.")
print("📌 Mini-batch: equilibrio entre ruido y eficiencia.")

---
## 6. Conexiones IA

### 6.1. Deep Learning
PyTorch y TensorFlow usan diferenciación automática para entrenar redes con millones de parámetros. El mismo principio aplica a arquitecturas complejas.

### 6.2. Generative AI (LLMs)
El fine-tuning de modelos como GPT o LLaMA usa optimizadores como **Adam**, que es una evolución del SGD básico.

### 6.3. Reinforcement Learning
Algoritmos como **Deep Q-Learning** actualizan la red Q usando gradiente descendente.

In [ ]:
# Ejemplo: usando Adam en PyTorch
model_adam = nn.Linear(1, 1)
with torch.no_grad():
    model_adam.weight.fill_(0.0)
    model_adam.bias.fill_(0.0)
optimizer_adam = optim.Adam(model_adam.parameters(), lr=0.1)

loss_adam = []
for epoch in range(50):
    y_pred = model_adam(X_t)
    loss = criterion(y_pred, y_t)
    loss_adam.append(loss.item())
    optimizer_adam.zero_grad()
    loss.backward()
    optimizer_adam.step()

plt.figure(figsize=(10, 6))
plt.plot(loss_pt_batch, 'b-', label='SGD')
plt.plot(loss_adam, 'r-', label='Adam')
plt.xlabel('Época')
plt.ylabel('MSE')
plt.title('Comparación: SGD vs Adam')
plt.legend()
plt.grid(True)
plt.show()

---
## Ejercicios para el Estudiante

1.  **Diferentes batch sizes:** Experimenta con batch sizes de 2, 8, 32, 64. ¿Cómo afecta a la suavidad de la curva de pérdida y al tiempo de cómputo?

2.  **Optimizadores:** Prueba los optimizadores **RMSprop** y **Adagrad** en PyTorch. Compáralos con SGD y Adam.

3.  **Inicialización:** Prueba diferentes inicializaciones de pesos (ceros, unos, valores aleatorios grandes). ¿Cómo afecta a la convergencia?

4.  **Problema no lineal:** Genera datos con una relación no lineal (ej. $y = x^2 + \epsilon$) y entrena un modelo lineal. ¿Qué observas en la pérdida? ¿Cómo podrías mejorarlo?

5.  **Reflexión:** ¿Por qué crees que Adam es el optimizador por defecto en muchos proyectos de deep learning? Investiga sus ventajas sobre SGD.

---
## Conclusiones de la Sesión

*   Implementamos **descenso de gradiente desde cero** con NumPy, comprendiendo cada paso del algoritmo.
*   Utilizamos **PyTorch** con `autograd` y `optim` para automatizar el cálculo de gradientes y la actualización de parámetros.
*   Comparamos ambas implementaciones, observando que los resultados son prácticamente idénticos, pero PyTorch simplifica enormemente el código.
*   Experimentamos con diferentes **tasas de aprendizaje**, viendo su efecto en la convergencia (lenta, óptima, divergente).
*   Exploramos las variantes del descenso de gradiente: **Batch**, **Stochastic (SGD)** y **Mini-batch**, analizando sus características.
*   Conectamos estos conceptos con aplicaciones reales en **Deep Learning**, **Generative AI** (fine-tuning de LLMs con Adam) y **Reinforcement Learning**.

¡Ahora dominas tanto la implementación manual como la automática del gradiente descendente!